# 03 — Capa Gold: constelación analítica

## Objetivo

Este notebook sirve como guion técnico y visual para la exposición. Todas las
comprobaciones usan los artefactos completos del proyecto; las vistas de pocas filas
son únicamente para mostrar el esquema, no son el conjunto usado por el pipeline.

**QUÉ EXPLICAR:** primero diga qué problema resuelve esta etapa, después muestre la
evidencia y termine conectándola con la siguiente capa.

## Preparación

Ejecútelo desde el contenedor `spark` con la raíz `/workspace`. La celda también
encuentra el repositorio cuando el notebook se abre desde la carpeta local.

In [ ]:
from pathlib import Path
import json

# Encontrar la raíz permite usar el mismo notebook en Docker o en el repositorio local.
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace')]
ROOT = next(
    path for path in candidates
    if (path / 'config' / 'pipeline.yaml').exists()
)
DATA = ROOT / 'data'
EXPORTS = ROOT / 'exports'
print(f'Raíz del proyecto: {ROOT}')

## Pasos

### 1. Nueve tablas exigidas y seis salidas auxiliares

Las nueve tablas base cubren 3 descriptivas, 3 diagnósticas y 3 predictivas. Las seis
tablas adicionales contienen métricas, perfiles y matriz de confusión.

**QUÉ EXPLICAR:** es una constelación (galaxy schema), porque varios hechos comparten
fecha, hora, servicio, zona, pago y segmento.

In [ ]:
import pyarrow.parquet as pq

gold = DATA / 'gold'
tables = sorted(path for path in gold.iterdir() if path.is_dir())

# Leer todos los metadatos da el conteo exacto sin cargar millones de filas en RAM.
counts = {}
for table in tables:
    parquet_files = list(table.rglob('*.parquet'))
    counts[table.name] = sum(pq.ParquetFile(path).metadata.num_rows for path in parquet_files)
for name, rows in counts.items():
    print(f'{name:42s} {rows:>12,}')

### 2. Clasificación por propósito

In [ ]:
base = {
    'descriptivo': [n for n in counts if n.startswith('descriptive_')],
    'diagnóstico': [n for n in counts if n.startswith('diagnostic_')],
    'predictivo': [n for n in counts if n in {
        'model_timeseries_daily', 'model_segmentation_zones',
        'model_classification_demand'}],
}
print(json.dumps(base, indent=2, ensure_ascii=False))
assert [len(base[k]) for k in base] == [3, 3, 3]

## Comprobaciones

In [ ]:
report = json.loads((EXPORTS / 'verification_report.json').read_text(encoding='utf-8'))
check = next(c for c in report['checks'] if c['name'] == 'nine_gold_tables')
assert check['passed'] and check['details']['tables'] == 9
assert len(counts) >= 15
print('Gold base:', check['details'])

## Siguiente paso

Las tablas `model_*` alimentan GBT, K-Means y Random Forest; el siguiente notebook
presenta sus métricas y artefactos persistidos.